# Task 2: checkpoint aggregation, model selection, and prediction
This notebook is the sole owner of final Task 2 evaluation and exports. It reads setup evidence and completed model checkpoints; worker CSV/JSON outputs are not inputs.

In [ ]:
%matplotlib inline
import json, shutil, sys
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import confusion_matrix
from torchvision.models import densenet121, efficientnet_b0
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
if REPO_ROOT is None: raise FileNotFoundError('Could not find repository root')
sys.path.insert(0, str(REPO_ROOT))
from src.preprocessing import TEST_IMAGE_DIR, load_image_array
from src.task2_utils import (TASK2_BASELINE_SCORES_PATH, TASK2_CHECKPOINT_DIR, TASK2_FIGURE_DIR, TASK2_MODEL_DIR,
    TASK2_OUTPUT_DIR, NeuralTrainer, ensure_task2_directories, evaluate_predictions,
    extract_visual_features, load_checkpoint_candidate, per_class_table,
    prepared_namespace, random_forest_paths)
ensure_task2_directories()
sns.set_theme(style='whitegrid', context='notebook')


## Load and validate checkpoint evidence

In [ ]:
data = prepared_namespace()
setup = data.config
CLASSES, TARGET, FINGERPRINT = data.classes, data.target, data.fingerprint
validation_ids = data.validation_ids
def validate_arrays(label, ids, truth, scores):
    if not np.array_equal(np.asarray(ids, dtype=str), validation_ids): raise ValueError(f'{label}: validation-ID ordering mismatch')
    if not np.array_equal(np.asarray(truth, dtype=int), data.y_val): raise ValueError(f'{label}: validation truth mismatch')
    scores = np.asarray(scores)
    if scores.shape != (len(validation_ids), len(CLASSES)): raise ValueError(f'{label}: invalid score shape {scores.shape}')
    if not np.isfinite(scores).all(): raise ValueError(f'{label}: non-finite scores')
    return scores
baseline_path = TASK2_BASELINE_SCORES_PATH
if not baseline_path.exists(): raise FileNotFoundError(f'Missing {baseline_path}; rerun Notebook 01')
with np.load(baseline_path, allow_pickle=False) as saved:
    if str(saved['fingerprint'].item()) != FINGERPRINT: raise ValueError('Baseline fingerprint mismatch')
    if list(saved['classes'].astype(str)) != list(CLASSES): raise ValueError('Baseline class ordering mismatch')
    baseline_scores = validate_arrays('Baseline', saved['validation_ids'], saved['true_indices'], saved['majority_scores'])
rf_model_path, rf_scores_path = random_forest_paths()
if not (rf_model_path.exists() and rf_scores_path.exists()): raise FileNotFoundError('Missing Random Forest checkpoint pair; run Notebook 02')
with np.load(rf_scores_path, allow_pickle=False) as saved:
    if str(saved['fingerprint'].item()) != FINGERPRINT: raise ValueError('Random Forest fingerprint mismatch')
    if list(saved['classes'].astype(str)) != list(CLASSES): raise ValueError('Random Forest class ordering mismatch')
    rf_scores = validate_arrays('Random Forest', saved['validation_ids'], saved['true_indices'], saved['scores'])
    rf_config = json.loads(str(saved['model_config_json'].item()))
efficientnet_record = load_checkpoint_candidate('efficientnet_b0', fingerprint=FINGERPRINT, classes=CLASSES, validation_ids=validation_ids)
densenet_record = load_checkpoint_candidate('densenet121', fingerprint=FINGERPRINT, classes=CLASSES, validation_ids=validation_ids)
print('Validated all checkpoint evidence for run', FINGERPRINT)


## Consistent comparison and winner selection

In [ ]:
def softmax(x):
    shifted = x - x.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
candidates = [
    {'name': 'Baseline: majority class', 'scores': baseline_scores, 'probabilities': baseline_scores, 'kind': 'baseline'},
    {'name': 'Random Forest', 'scores': rf_scores, 'probabilities': rf_scores, 'kind': 'rf', 'model_config': rf_config, 'checkpoint_path': rf_model_path},
    {**efficientnet_record, 'name': 'EfficientNet-B0', 'probabilities': softmax(efficientnet_record['scores']), 'kind': 'neural'},
    {**densenet_record, 'name': 'DenseNet-121', 'probabilities': softmax(densenet_record['scores']), 'kind': 'neural'},
]
rows = [evaluate_predictions(data.y_val, item['scores'].argmax(1), item['scores'], item['name']) for item in candidates]
model_comparison = pd.DataFrame(rows).sort_values('Macro-F1', ascending=False).reset_index(drop=True)
trained = model_comparison[model_comparison['Model'] != 'Baseline: majority class']
FINAL_NAME = trained.iloc[0]['Model']
winner = next(item for item in candidates if item['name'] == FINAL_NAME)
final_scores, final_probabilities = winner['scores'], winner['probabilities']
y_true, y_pred = data.y_val, final_scores.argmax(1)
display(model_comparison)
print('Selected by Macro-F1:', FINAL_NAME)


## Selected-model diagnostics

In [ ]:
season_results = per_class_table(y_true, y_pred, CLASSES)
display(season_results)
matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)), normalize='true')
plt.figure(figsize=(7, 6)); sns.heatmap(matrix, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, vmin=0, vmax=1)
plt.title(f'{FINAL_NAME}: row-normalised confusion matrix'); plt.xlabel('Predicted season'); plt.ylabel('True season'); plt.tight_layout()
plt.savefig(TASK2_FIGURE_DIR / 'confusion_matrix.png', dpi=180, bbox_inches='tight'); plt.show()
def calibration(probabilities, truth, n_bins=10):
    confidence, predicted = probabilities.max(1), probabilities.argmax(1)
    rows, ece = [], 0.0
    for lower, upper in zip(np.linspace(0,1,n_bins+1)[:-1], np.linspace(0,1,n_bins+1)[1:]):
        chosen = (confidence > lower) & (confidence <= upper)
        if chosen.any():
            accuracy, mean_confidence = (predicted[chosen] == truth[chosen]).mean(), confidence[chosen].mean()
            ece += chosen.mean() * abs(accuracy - mean_confidence)
            rows.append({'confidence': mean_confidence, 'accuracy': accuracy, 'count': int(chosen.sum())})
    return ece, pd.DataFrame(rows)
ece, calibration_frame = calibration(final_probabilities, y_true)
plt.figure(figsize=(5.5,5)); plt.plot([0,1],[0,1],'--',label='Perfect calibration'); plt.plot(calibration_frame['confidence'], calibration_frame['accuracy'], marker='o', label=FINAL_NAME)
plt.xlabel('Mean confidence'); plt.ylabel('Observed accuracy'); plt.title(f'Validation calibration (ECE={ece:.4f})'); plt.legend(); plt.tight_layout()
plt.savefig(TASK2_FIGURE_DIR / 'calibration.png', dpi=180, bbox_inches='tight'); plt.show()
plt.figure(figsize=(7,4))
for label, record in [('EfficientNet-B0', efficientnet_record), ('DenseNet-121', densenet_record)]:
    history = pd.DataFrame(record['history']); plt.plot(history['epoch'], history['val macro-F1'], marker='o', label=label)
plt.xlabel('Epoch'); plt.ylabel('Validation Macro-F1'); plt.title('Neural training curves'); plt.legend(); plt.tight_layout()
plt.savefig(TASK2_FIGURE_DIR / 'training_curves.png', dpi=180, bbox_inches='tight'); plt.show()
efficientnet_count = efficientnet_b0(weights=None); efficientnet_count.classifier[1] = nn.Linear(efficientnet_count.classifier[1].in_features, len(CLASSES))
densenet_count = densenet121(weights=None); densenet_count.classifier = nn.Linear(densenet_count.classifier.in_features, len(CLASSES))
parameter_counts = {'Random Forest': np.nan, 'EfficientNet-B0': sum(p.numel() for p in efficientnet_count.parameters()), 'DenseNet-121': sum(p.numel() for p in densenet_count.parameters())}
deployment = pd.DataFrame([{'Model': item['name'], 'Parameters': parameter_counts[item['name']], 'Checkpoint size (MB)': item['checkpoint_path'].stat().st_size/1e6} for item in candidates if item['kind'] != 'baseline'])
display(deployment)


## Consolidated exports and selected-model package

In [ ]:
model_comparison.to_csv(TASK2_MODEL_DIR / 'task2_results.csv', index=False)
season_results.to_csv(TASK2_MODEL_DIR / 'task2_per_class_results.csv', index=False)
summary_rows = []
for record in (efficientnet_record, densenet_record):
    history = pd.DataFrame(record['history'])
    best = history.loc[history['val macro-F1'].idxmax()]
    summary_rows.append({'Model': record['name'], 'Best epoch': int(best['epoch']), 'Best Macro-F1': best['val macro-F1'], 'Training seconds': history['seconds'].sum(), 'Epochs completed': len(history)})
training_summary = pd.DataFrame(summary_rows)
training_summary.to_csv(TASK2_MODEL_DIR / 'task2_training_summary.csv', index=False)
winner_metric = float(trained.iloc[0]['Macro-F1'])
package_metadata = {'selected_model': FINAL_NAME, 'fingerprint': FINGERPRINT, 'model_config': winner.get('model_config', {}), 'classes': CLASSES, 'image_target_size': setup['image_target_size'], 'normalisation_mean': setup['normalisation_mean'], 'normalisation_std': setup['normalisation_std'], 'selection_metric': 'Macro-F1', 'validation_value': winner_metric}
if winner['kind'] == 'rf':
    selected_model_path = TASK2_MODEL_DIR / 'task2_model.joblib'
    joblib.dump({'model': joblib.load(rf_model_path), **package_metadata}, selected_model_path)
else:
    selected_model_path = TASK2_MODEL_DIR / 'task2_model.pt'
    NeuralTrainer._save({'state_dict': winner['blob']['state_dict'], **package_metadata}, selected_model_path)
print('Saved final tables, figures, and selected model:', selected_model_path)


## Final test inference and durable predictions

In [ ]:
template = pd.read_csv(REPO_ROOT / 'datasets' / 'test' / 'styles_prediction.csv')
test_paths = [Path(TEST_IMAGE_DIR) / f'{image_id}.jpg' for image_id in template['id']]
missing = [path for path in test_paths if not path.exists()]
if missing: raise FileNotFoundError(f'{len(missing)} test images are missing')
if winner['kind'] == 'rf':
    model = joblib.load(rf_model_path)
    features = np.vstack([extract_visual_features(load_image_array(path, tuple(setup['image_target_size']), False)) for path in test_paths])
    test_scores = model.predict_proba(features)
    predicted = model.classes_[test_scores.argmax(1)]
else:
    if FINAL_NAME == 'EfficientNet-B0':
        model = efficientnet_b0(weights=None); model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
    else:
        model = densenet121(weights=None); model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    model.load_state_dict(winner['blob']['state_dict'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model = model.to(device).eval()
    mean = torch.tensor(setup['normalisation_mean'], device=device).view(1,3,1,1); std = torch.tensor(setup['normalisation_std'], device=device).view(1,3,1,1)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(test_paths), 256):
            arrays = np.stack([load_image_array(path, tuple(setup['image_target_size']), False) for path in test_paths[start:start+256]])
            images = torch.from_numpy(arrays).to(device).permute(0,3,1,2).float() / 255
            chunks.append(model((images-mean)/std).cpu())
    test_scores = torch.cat(chunks).numpy(); predicted = test_scores.argmax(1)
predictions = template.copy(); predictions[TARGET] = [CLASSES[int(index)] for index in predicted]
prediction_path = TASK2_OUTPUT_DIR / 'task2_predictions.csv'
predictions.to_csv(prediction_path, index=False)
np.save(TASK2_OUTPUT_DIR / 'task2_test_scores.npy', test_scores)
print('Saved predictions:', prediction_path)
display(predictions.head())
